In [1]:
!pip install rank_bm25 sentencepiece

In [2]:
import os
import json
from tqdm.auto import tqdm

# --- EXACT KAGGLE PATHS ---
CASES_DIR = "/kaggle/input/datasets/somdevganguli/coliee-cases/cases" 
LABELS_FILE = "/kaggle/input/datasets/somdevganguli/coliee-train-labels/task2_train_labels_2026.json"

# 1. Load the ground-truth labels
with open(LABELS_FILE, "r", encoding="utf-8") as f:
    train_labels = json.load(f)

dataset = []

# 2. Iterate through every query case
for query_id, positive_paras in tqdm(train_labels.items(), desc="Loading COLIEE Cases"):
    case_folder = os.path.join(CASES_DIR, query_id)
    if not os.path.exists(case_folder): 
        continue
        
    fragment_path = os.path.join(case_folder, "entailed_fragment.txt")
    base_case_path = os.path.join(case_folder, "base_case.txt")
    paragraphs_dir = os.path.join(case_folder, "paragraphs")
    
    try:
        with open(fragment_path, "r", encoding="utf-8") as f:
            entailed_fragment = f.read().strip()
    except FileNotFoundError: 
        continue 
        
    try:
        with open(base_case_path, "r", encoding="utf-8") as f:
            base_case = f.read().strip()
    except FileNotFoundError: 
        base_case = ""
        
    candidates = {}
    if os.path.exists(paragraphs_dir):
        for para_file in os.listdir(paragraphs_dir):
            if para_file.endswith(".txt"):
                with open(os.path.join(paragraphs_dir, para_file), "r", encoding="utf-8") as f:
                    candidates[para_file] = f.read().strip()
                    
    dataset.append({
        "query_id": query_id,
        "entailed_fragment": entailed_fragment,
        "base_case": base_case,
        "candidates": candidates,
        "ground_truth": positive_paras
    })

print(f"\n✅ Successfully loaded {len(dataset)} query cases into memory!")

Loading COLIEE Cases:   0%|          | 0/925 [00:00<?, ?it/s]


✅ Successfully loaded 925 query cases into memory!


In [3]:
import re
import random
import pandas as pd
from sklearn.model_selection import train_test_split
from rank_bm25 import BM25Okapi

print("🧠 Initializing Context Extractor and Data Splits...")

# 80/20 Shuffled Split (Seed 42 for reproducibility)
all_qids = [case["query_id"] for case in dataset]
train_qids, val_qids = train_test_split(all_qids, test_size=0.2, random_state=42)

def compress_base_case(base_case, fragment, top_k=2):
    """Extract top-k sentences from base_case most relevant to the fragment."""
    if not base_case:
        return ""
    # Split by basic punctuation
    sentences = re.split(r'(?<=[.!?])\s+', base_case)
    sentences = [s.strip() for s in sentences if len(s.strip().split()) > 5]
    if not sentences:
        return ""
    
    tokenized = [s.lower().split() for s in sentences]
    bm25 = BM25Okapi(tokenized)
    scores = bm25.get_scores(fragment.lower().split())
    
    # Get top_k highest scoring sentences and maintain original order
    top_indices = scores.argsort()[-top_k:][::-1]
    return " ".join([sentences[i] for i in sorted(top_indices)])

def build_enriched_query(case):
    """Build query = [CONTEXT] + [FRAGMENT] for Retrieval only."""
    fragment = case["entailed_fragment"]
    context = compress_base_case(case["base_case"], fragment)
    return f"[CONTEXT] {context} [FRAGMENT] {fragment}" if context else fragment

print(f"✅ Split Complete! Train: {len(train_qids)} | Val: {len(val_qids)}")

🧠 Initializing Context Extractor and Data Splits...
✅ Split Complete! Train: 740 | Val: 185


In [4]:
import torch
import string
import gc
import json
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util
from tqdm.auto import tqdm

# --- CONFIGURATION ---
MAX_CANDIDATES = 80 # Increased from 30
W_BGE = 1.0
W_BM25 = 0.2 # Bumped slightly for legal lexical cues
CHECKPOINT_PATH = "./hybrid_wrf_decoupled_checkpoint.json"

device = 'cuda' if torch.cuda.is_available() else 'cpu'
bge_model = SentenceTransformer('BAAI/bge-m3', device=device, model_kwargs={"torch_dtype": torch.float16})

def tokenize(text):
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    return text.split()

def weighted_reciprocal_rank_fusion(bge_ranked_list, bm25_ranked_list, k=60, w_bge=1.0, w_bm25=0.2):
    scores = {}
    for rank, doc in enumerate(bge_ranked_list):
        scores[doc] = scores.get(doc, 0.0) + w_bge * (1.0 / (k + rank + 1))
    for rank, doc in enumerate(bm25_ranked_list):
        scores[doc] = scores.get(doc, 0.0) + w_bm25 * (1.0 / (k + rank + 1))
    return sorted(scores.keys(), key=lambda x: scores[x], reverse=True)

hybrid_map = {}
hits_at_k = 0

for case in tqdm(dataset, desc=f"Decoupled wRRF (Top {MAX_CANDIDATES})"):
    q_id = case["query_id"]
    ground_truth = case["ground_truth"]
    candidates = case["candidates"]
    
    # THE DECOUPLING: Use Enriched Query for Stage 1
    enriched_query_text = build_enriched_query(case)
    
    para_filenames = list(candidates.keys())
    para_texts = list(candidates.values())
    
    if not para_texts:
        hybrid_map[q_id] = []
        continue
        
    # BM25 SCORING
    tokenized_corpus = [tokenize(doc) for doc in para_texts]
    bm25 = BM25Okapi(tokenized_corpus)
    tokenized_query = tokenize(enriched_query_text)
    bm25_scores = bm25.get_scores(tokenized_query)
    scored_bm25 = sorted(zip(para_filenames, bm25_scores), key=lambda x: x[1], reverse=True)
    bm25_ranks = [f for f, s in scored_bm25]
    
    # BGE SCORING
    query_embedding = bge_model.encode(enriched_query_text, convert_to_tensor=True, show_progress_bar=False)
    passage_embeddings = bge_model.encode(para_texts, batch_size=32, convert_to_tensor=True, show_progress_bar=False)
    cosine_scores = util.cos_sim(query_embedding, passage_embeddings)[0] 
    
    scores_cpu = cosine_scores.cpu().numpy()
    ranked_indices = np.argsort(scores_cpu)[::-1]
    bge_ranks = [para_filenames[i] for i in ranked_indices]
    
    # FUSION
    fused_ranks = weighted_reciprocal_rank_fusion(bge_ranks, bm25_ranks, k=60, w_bge=W_BGE, w_bm25=W_BM25)
    
    dynamic_top_k = min(len(fused_ranks), MAX_CANDIDATES)
    top_k_hybrid = fused_ranks[:dynamic_top_k]
    hybrid_map[q_id] = top_k_hybrid
    
    if any(truth in top_k_hybrid for truth in ground_truth):
        hits_at_k += 1

with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
    json.dump(hybrid_map, f, indent=4)

del bge_model
gc.collect()
torch.cuda.empty_cache()

print(f"✅ wRRF Top-{MAX_CANDIDATES} Complete! Hybrid Recall Ceiling: {hits_at_k/len(dataset):.4f}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Decoupled wRRF (Top 80):   0%|          | 0/925 [00:00<?, ?it/s]

✅ wRRF Top-80 Complete! Hybrid Recall Ceiling: 0.8919


In [5]:
from datasets import Dataset as HFDataset

def create_bucketed_training_dataset(qids):
    records = []
    for case in dataset:
        if case["query_id"] not in qids: continue
        
        # CRITICAL DECOUPLING: Plain fragment to save LegalBERT token budget
        query = case["entailed_fragment"] 
        ground_truth = set(case["ground_truth"])
        candidates = case["candidates"]
        hybrid_top = hybrid_map.get(case["query_id"], [])
        
        # 1. Positives
        for gt_file in ground_truth:
            if gt_file in candidates:
                records.append({"query": query, "paragraph": candidates[gt_file], "label": 1})
        
        # 2. Bucketed Negatives (from wRRF results)
        negatives = [f for f in hybrid_top if f not in ground_truth and f in candidates]
        
        if len(negatives) > 0:
            # Hard Negatives (Ranks 1-20)
            hard = negatives[:20]
            for n in hard[:6]:
                records.append({"query": query, "paragraph": candidates[n], "label": 0})
            
            # Semi-Hard Negatives (Ranks 21-60)
            semi_hard = negatives[20:60]
            if semi_hard:
                sample_size = min(4, len(semi_hard))
                for n in random.sample(semi_hard, sample_size):
                    records.append({"query": query, "paragraph": candidates[n], "label": 0})
            
            # Easy Negatives (Ranks 60+)
            easy = negatives[60:]
            if easy:
                sample_size = min(2, len(easy))
                for n in random.sample(easy, sample_size):
                    records.append({"query": query, "paragraph": candidates[n], "label": 0})

    return pd.DataFrame(records)

train_df = create_bucketed_training_dataset(train_qids)
hf_train_dataset = HFDataset.from_pandas(train_df)
print(f"✅ Generated {len(hf_train_dataset)} highly curated training pairs.")

✅ Generated 7285 highly curated training pairs.


In [6]:
import torch.nn as nn
import torch.nn.functional as F
import math
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
from transformers.modeling_outputs import SequenceClassifierOutput

MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(ex):
    return tokenizer(ex["query"], ex["paragraph"], padding="max_length", truncation=True, max_length=512)

tokenized_train = hf_train_dataset.map(tokenize_fn, batched=True).remove_columns(["query", "paragraph"])

class PMAPooling(nn.Module):
    def __init__(self, hidden_size, num_heads):
        super(PMAPooling, self).__init__()
        self.num_heads = num_heads
        self.d_model = hidden_size
        self.q = nn.Parameter(torch.randn(num_heads, hidden_size))
        self.k = nn.Linear(hidden_size, hidden_size)
        self.v = nn.Linear(hidden_size, hidden_size)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, hidden_states, attention_mask):
        k = self.k(hidden_states)
        v = self.v(hidden_states)
        scores = torch.einsum('hd,bsd->bhs', self.q, k) / math.sqrt(self.d_model)
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(1).expand(-1, self.num_heads, -1)
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = self.softmax(scores)
        pooled_output = torch.matmul(attn_weights, v)
        return pooled_output.view(hidden_states.size(0), -1)

class PMALegalCrossEncoder(nn.Module):
    def __init__(self, model_name, num_labels=2, num_heads=4):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.config = self.bert.config
        self.pma = PMAPooling(hidden_size=self.config.hidden_size, num_heads=num_heads)
        self.classifier = nn.Linear(self.config.hidden_size * num_heads, num_labels)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, labels=None, **kwargs):
        outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled = self.pma(outputs.last_hidden_state, attention_mask)
        logits = self.classifier(pooled)
        return SequenceClassifierOutput(logits=logits)

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss) 
        F_loss = self.alpha * (1 - pt)**self.gamma * ce_loss
        return torch.mean(F_loss) if self.reduction == 'mean' else F_loss

class FocalLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fct = FocalLoss(alpha=0.75, gamma=2.0)
        loss = loss_fct(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

model = PMALegalCrossEncoder(MODEL_NAME, num_labels=2, num_heads=4).to("cuda")

args = TrainingArguments(
    output_dir="./legalbert_pma_focal_run",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    warmup_ratio=0.1, 
    fp16=True, 
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    save_strategy="no"
)

trainer = FocalLossTrainer(model=model, args=args, train_dataset=tokenized_train)

print("\n🚀 Fine-Tuning LegalBERT with PMA + Focal Loss...")
trainer.train()
print("✅ LegalBERT Training Complete.")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/7285 [00:00<?, ? examples/s]

...

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



🚀 Fine-Tuning LegalBERT with PMA + Focal Loss...


Step,Training Loss
50,0.162635
100,0.117968
150,0.097668
200,0.109060
250,0.087550
300,0.072891
350,0.076792
400,0.078478
450,0.080093
500,0.059780


✅ LegalBERT Training Complete.


In [7]:
model.eval()
val_predictions = {}
total_relevant = 0

val_cases = [c for c in dataset if c["query_id"] in val_qids]

print("\n🧠 Running Neural Inference on Validation Fold...")
for case in tqdm(val_cases, desc="Scoring Candidates"):
    q_id = case["query_id"]
    # CRITICAL: Plain Fragment Inference
    query = case["entailed_fragment"] 
    ground_truth = set(case["ground_truth"])
    candidates = case["candidates"]
    
    hybrid_top = hybrid_map.get(q_id, []) 
    valid_fnames = [f for f in hybrid_top if f in candidates]
    
    if not valid_fnames:
        total_relevant += len(ground_truth)
        val_predictions[q_id] = {"scored": [], "ground_truth": ground_truth}
        continue
        
    texts = [candidates[f] for f in valid_fnames]
    all_probs = []
    
    for i in range(0, len(texts), 8):
        batch = texts[i:i+8]
        inputs = tokenizer([query]*len(batch), batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to("cuda")
        
        with torch.no_grad(), torch.amp.autocast('cuda'):
            logits = model(**inputs).logits
            probs = F.softmax(logits, dim=-1)[:, 1].cpu().tolist()
            all_probs.extend(probs)
            
    scored = list(zip(valid_fnames, all_probs))
    scored.sort(key=lambda x: x[1], reverse=True)
    val_predictions[q_id] = {"scored": scored, "ground_truth": ground_truth}
    total_relevant += len(ground_truth)

# --- 3-GATE SAFE MAX-GAP ---
def safe_max_gap(scored_candidates, tau_none=0.10, tau_low=0.20, tau_accept=0.35, gap_min=0.06, ratio_min=1.30):
    if not scored_candidates: return []
    
    p1 = scored_candidates[0][1]
    # Gate 1: Absolute noise
    if p1 < tau_none: return [scored_candidates[0][0]]
        
    # Gate 2: Plateau Guard
    p2 = scored_candidates[1][1] if len(scored_candidates) > 1 else 0
    if p1 < tau_low and (p1 - p2) < 0.03: return [scored_candidates[0][0]]

    max_gap = 0
    best_cut = 1
    search_limit = min(len(scored_candidates), 8)

    for i in range(search_limit - 1):
        gap = scored_candidates[i][1] - scored_candidates[i+1][1]
        if gap > max_gap:
            max_gap = gap
            best_cut = i + 1

    # Gate 3: Acceptance Gate
    cut_prob = scored_candidates[best_cut-1][1]
    next_prob = scored_candidates[best_cut][1] if best_cut < len(scored_candidates) else 0
    
    is_valid_gap = (cut_prob >= tau_accept) and (max_gap >= gap_min) and ((cut_prob / (next_prob + 1e-9)) >= ratio_min)
    
    if is_valid_gap:
        return [f for f, p in scored_candidates[:best_cut]]
    else:
        return [scored_candidates[0][0]]

# --- EVALUATION ---
total_tp, total_retrieved = 0, 0
for q_id, data in val_predictions.items():
    predicted = safe_max_gap(data["scored"])
    pred_set = set(predicted)
    total_tp += len(pred_set.intersection(data["ground_truth"]))
    total_retrieved += len(pred_set)

p = total_tp / total_retrieved if total_retrieved > 0 else 0
r = total_tp / total_relevant if total_relevant > 0 else 0
f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0

print("\n" + "="*50)
print(f"🏆 FINAL DECOUPLED PIPELINE RESULTS (SAFE MAX-GAP) 🏆")
print("="*50)
print(f"Precision:  {p:.4f}")
print(f"Recall:     {r:.4f}")
print(f"F1-Measure: {f1:.4f}")
print("="*50)


🧠 Running Neural Inference on Validation Fold...


Scoring Candidates:   0%|          | 0/185 [00:00<?, ?it/s]


🏆 FINAL DECOUPLED PIPELINE RESULTS (SAFE MAX-GAP) 🏆
Precision:  0.5228
Recall:     0.3066
F1-Measure: 0.3865


In [8]:
# ═══════════════════════════════════════════════════════════════════════════════
# B2: MICRO vs. MACRO AVERAGING VERIFICATION
# ═══════════════════════════════════════════════════════════════════════════════
# Reviewer 2 flagged that the paper claims "macro-averaged" but the code
# computes micro-averaged metrics. This cell computes BOTH and compares.
#
# Paper reference: Section 4.1 (line 287) — now corrected to "micro-averaged"
# ═══════════════════════════════════════════════════════════════════════════════

import numpy as np

def compute_micro_metrics(val_predictions, threshold_fn):
    """Micro-averaged: global TP / global retrieved, global TP / global relevant."""
    total_tp, total_retrieved, total_relevant = 0, 0, 0
    for q_id, data in val_predictions.items():
        predicted = threshold_fn(data["scored"])
        pred_set = set(predicted)
        gt = data["ground_truth"]
        total_tp += len(pred_set & gt)
        total_retrieved += len(pred_set)
        total_relevant += len(gt)
    p = total_tp / total_retrieved if total_retrieved > 0 else 0
    r = total_tp / total_relevant if total_relevant > 0 else 0
    f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0
    return p, r, f1

def compute_macro_metrics(val_predictions, threshold_fn):
    """Macro-averaged: per-query P, R, F1, then average across queries."""
    precisions, recalls, f1s = [], [], []
    for q_id, data in val_predictions.items():
        predicted = threshold_fn(data["scored"])
        pred_set = set(predicted)
        gt = data["ground_truth"]
        tp = len(pred_set & gt)
        p = tp / len(pred_set) if len(pred_set) > 0 else 0
        r = tp / len(gt) if len(gt) > 0 else 0
        f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0
        precisions.append(p)
        recalls.append(r)
        f1s.append(f1)
    return np.mean(precisions), np.mean(recalls), np.mean(f1s)

# --- Threshold functions ---
def threshold_3gate(scored):
    return safe_max_gap(scored)

def threshold_static(t):
    def _fn(scored):
        if not scored:
            return []
        selected = [f for f, p in scored if p >= t]
        return selected if selected else [scored[0][0]]
    return _fn

def threshold_top1(scored):
    return [scored[0][0]] if scored else []

# --- Compare micro vs. macro for key configurations ---
configs = [
    ("3-Gate Max-Gap", threshold_3gate),
    ("Static 0.35", threshold_static(0.35)),
    ("Static 0.40", threshold_static(0.40)),
    ("Static 0.45", threshold_static(0.45)),
    ("Static 0.50", threshold_static(0.50)),
    ("Top-1 (Argmax)", threshold_top1),
]

print("=" * 85)
print("B2: MICRO vs. MACRO AVERAGING COMPARISON")
print("=" * 85)
print(f"{'Strategy':<22} | {'Micro-P':>8} {'Micro-R':>8} {'Micro-F1':>9} | {'Macro-P':>8} {'Macro-R':>8} {'Macro-F1':>9}")
print("-" * 85)

for name, fn in configs:
    mi_p, mi_r, mi_f1 = compute_micro_metrics(val_predictions, fn)
    ma_p, ma_r, ma_f1 = compute_macro_metrics(val_predictions, fn)
    print(f"{name:<22} | {mi_p:>8.4f} {mi_r:>8.4f} {mi_f1:>9.4f} | {ma_p:>8.4f} {ma_r:>8.4f} {ma_f1:>9.4f}")

print("=" * 85)
print("\n✅ VERDICT: Paper Section 4.1 now correctly states 'micro-averaged'.")
print("   The existing evaluation code computes micro-averaged metrics (global TP / global retrieved).")
print("   Macro numbers shown above for reference — use micro for paper consistency.")

B2: MICRO vs. MACRO AVERAGING COMPARISON
Strategy               |  Micro-P  Micro-R  Micro-F1 |  Macro-P  Macro-R  Macro-F1
-------------------------------------------------------------------------------------
3-Gate Max-Gap         |   0.5228   0.3066    0.3865 |   0.5700   0.5995    0.5579
Static 0.35            |   0.4149   0.3260    0.3651 |   0.4861   0.6338    0.5101
Static 0.40            |   0.4662   0.3187    0.3786 |   0.5166   0.6176    0.5289
Static 0.45            |   0.4924   0.3139    0.3834 |   0.5353   0.6140    0.5410
Static 0.50            |   0.5292   0.3090    0.3902 |   0.5601   0.6032    0.5560
Top-1 (Argmax)         |   0.6000   0.2701    0.3725 |   0.6000   0.5356    0.5544

✅ VERDICT: Paper Section 4.1 now correctly states 'micro-averaged'.
   The existing evaluation code computes micro-averaged metrics (global TP / global retrieved).
   Macro numbers shown above for reference — use micro for paper consistency.


In [ ]:
# =============================================================================
# COMPONENT-WISE ABLATION ON TEST DATA (100 Queries)
# =============================================================================
# This cell:
#   1. Loads the COLIEE 2026 test data (100 queries) + ground-truth labels
#   2. Runs wRRF retrieval (BGE-M3 + BM25) on test queries
#   3. Trains 6 ablation configurations on ALL 925 training queries
#   4. Evaluates each configuration on the 100 test queries
#
# Config matrix:
#   1. Vanilla BERT [CLS]         -- bert-base-uncased, [CLS], CE, standard neg, seed 42
#   2. LegalBERT [CLS]            -- legal-bert, [CLS], CE, standard neg, seed 42
#   3. LegalBERT + PMA            -- legal-bert, PMA, CE, standard neg, seed 42
#   4. LegalBERT + PMA + Focal    -- legal-bert, PMA, Focal, bucketed neg, seed 42
#   5. LegalBERT + PMA + Focal + Ensemble -- same as 4, seeds 42/43/44, Top-1
#   6. Full Pipeline (3-Gate)     -- same as 5, 3-Gate thresholding
#
# Estimated time: ~150-180 min total on a single T4
# =============================================================================

import os, json, re, gc, math, string, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset as HFDataset
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
from transformers.modeling_outputs import SequenceClassifierOutput
from sentence_transformers import SentenceTransformer, util
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm

# =====================================================================
# STEP 1: LOAD TEST DATA
# =====================================================================
# Adjust this path for your Kaggle dataset mount
TEST_CASES_DIR = "/kaggle/input/coliee-2026-task2-test-files/task2_test_files_2026"
TEST_LABELS_FILE = "/kaggle/input/coliee-2026-task2-test-labels/task2_test_labels_2026.json"

with open(TEST_LABELS_FILE, "r", encoding="utf-8") as f:
    test_labels_raw = json.load(f)

# Parse test labels: values are comma-separated strings -> convert to list
test_labels = {}
for qid, val in test_labels_raw.items():
    if isinstance(val, str):
        test_labels[qid] = [v.strip() for v in val.split(",")]
    else:
        test_labels[qid] = val

test_dataset = []
for query_id, positive_paras in tqdm(test_labels.items(), desc="Loading Test Cases"):
    case_folder = os.path.join(TEST_CASES_DIR, query_id)
    if not os.path.exists(case_folder):
        continue

    fragment_path = os.path.join(case_folder, "entailed_fragment.txt")
    base_case_path = os.path.join(case_folder, "base_case.txt")
    paragraphs_dir = os.path.join(case_folder, "paragraphs")

    try:
        with open(fragment_path, "r", encoding="utf-8") as f:
            entailed_fragment = f.read().strip()
    except FileNotFoundError:
        continue

    try:
        with open(base_case_path, "r", encoding="utf-8") as f:
            base_case = f.read().strip()
    except FileNotFoundError:
        base_case = ""

    candidates = {}
    if os.path.exists(paragraphs_dir):
        for para_file in os.listdir(paragraphs_dir):
            if para_file.endswith(".txt"):
                with open(os.path.join(paragraphs_dir, para_file), "r", encoding="utf-8") as f:
                    candidates[para_file] = f.read().strip()

    test_dataset.append({
        "query_id": query_id,
        "entailed_fragment": entailed_fragment,
        "base_case": base_case,
        "candidates": candidates,
        "ground_truth": positive_paras
    })

print(f"\n✅ Loaded {len(test_dataset)} test query cases!")

# =====================================================================
# STEP 2: wRRF RETRIEVAL ON TEST DATA
# =====================================================================
MAX_CANDIDATES = 80

device = 'cuda' if torch.cuda.is_available() else 'cpu'
bge_model = SentenceTransformer('BAAI/bge-m3', device=device, model_kwargs={"torch_dtype": torch.float16})

def tokenize_text(text):
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    return text.split()

def wrrf(bge_ranked, bm25_ranked, k=60, w_bge=1.0, w_bm25=0.2):
    scores = {}
    for rank, doc in enumerate(bge_ranked):
        scores[doc] = scores.get(doc, 0.0) + w_bge * (1.0 / (k + rank + 1))
    for rank, doc in enumerate(bm25_ranked):
        scores[doc] = scores.get(doc, 0.0) + w_bm25 * (1.0 / (k + rank + 1))
    return sorted(scores.keys(), key=lambda x: scores[x], reverse=True)

test_hybrid_map = {}
test_hits = 0

for case in tqdm(test_dataset, desc=f"Test wRRF (Top {MAX_CANDIDATES})"):
    q_id = case["query_id"]
    gt = case["ground_truth"]
    candidates = case["candidates"]

    enriched_query_text = build_enriched_query(case)

    para_filenames = list(candidates.keys())
    para_texts = list(candidates.values())

    if not para_texts:
        test_hybrid_map[q_id] = []
        continue

    # BM25
    tokenized_corpus = [tokenize_text(doc) for doc in para_texts]
    bm25 = BM25Okapi(tokenized_corpus)
    bm25_scores = bm25.get_scores(tokenize_text(enriched_query_text))
    scored_bm25 = sorted(zip(para_filenames, bm25_scores), key=lambda x: x[1], reverse=True)
    bm25_ranks = [f for f, s in scored_bm25]

    # BGE
    query_emb = bge_model.encode(enriched_query_text, convert_to_tensor=True, show_progress_bar=False)
    passage_embs = bge_model.encode(para_texts, batch_size=32, convert_to_tensor=True, show_progress_bar=False)
    cos_scores = util.cos_sim(query_emb, passage_embs)[0].cpu().numpy()
    bge_ranks = [para_filenames[i] for i in np.argsort(cos_scores)[::-1]]

    # Fusion
    fused = wrrf(bge_ranks, bm25_ranks, k=60, w_bge=1.0, w_bm25=0.2)
    top_k = fused[:min(len(fused), MAX_CANDIDATES)]
    test_hybrid_map[q_id] = top_k

    if any(t in top_k for t in gt):
        test_hits += 1

del bge_model; gc.collect(); torch.cuda.empty_cache()
print(f"✅ Test wRRF Complete! Recall Ceiling: {test_hits/len(test_dataset):.4f}")

# =====================================================================
# STEP 3: MODEL DEFINITIONS (self-contained)
# =====================================================================

class CLSCrossEncoder(nn.Module):
    """Standard [CLS]-pooled cross-encoder (no PMA)."""
    def __init__(self, model_name, num_labels=2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.config = self.bert.config
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, labels=None, **kwargs):
        outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled = self.dropout(outputs.last_hidden_state[:, 0, :])
        logits = self.classifier(pooled)
        return SequenceClassifierOutput(logits=logits)

class PMAPooling_T(nn.Module):
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_model = hidden_size
        self.q = nn.Parameter(torch.randn(num_heads, hidden_size))
        self.k = nn.Linear(hidden_size, hidden_size)
        self.v = nn.Linear(hidden_size, hidden_size)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, hidden_states, attention_mask):
        k = self.k(hidden_states)
        v = self.v(hidden_states)
        scores = torch.einsum('hd,bsd->bhs', self.q, k) / math.sqrt(self.d_model)
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(1).expand(-1, self.num_heads, -1)
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = self.softmax(scores)
        pooled_output = torch.matmul(attn_weights, v)
        return pooled_output.view(hidden_states.size(0), -1)

class PMACrossEncoder_T(nn.Module):
    def __init__(self, model_name, num_labels=2, num_heads=4):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.config = self.bert.config
        self.pma = PMAPooling_T(self.config.hidden_size, num_heads)
        self.classifier = nn.Linear(self.config.hidden_size * num_heads, num_labels)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, labels=None, **kwargs):
        outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled = self.pma(outputs.last_hidden_state, attention_mask)
        logits = self.classifier(pooled)
        return SequenceClassifierOutput(logits=logits)

class FocalLoss_T(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        return torch.mean(self.alpha * (1 - pt)**self.gamma * ce_loss)

class FocalTrainer_T(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = FocalLoss_T()(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

class CETrainer_T(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = F.cross_entropy(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

# =====================================================================
# STEP 4: DATASET BUILDERS
# =====================================================================

def create_standard_neg_train(neg_per_positive=12):
    """Standard (non-bucketed) random negatives from ALL 925 training queries."""
    records = []
    all_train_qids = set(str(c["query_id"]) for c in dataset)
    for case in dataset:
        query = case["entailed_fragment"]
        ground_truth = set(case["ground_truth"])
        candidates = case["candidates"]
        hybrid_top = hybrid_map.get(case["query_id"], [])

        for gt_file in ground_truth:
            if gt_file in candidates:
                records.append({"query": query, "paragraph": candidates[gt_file], "label": 1})

        negatives = [f for f in hybrid_top if f not in ground_truth and f in candidates]
        if negatives:
            sample_size = min(neg_per_positive, len(negatives))
            for n in random.sample(negatives, sample_size):
                records.append({"query": query, "paragraph": candidates[n], "label": 0})

    return pd.DataFrame(records)

def create_bucketed_train():
    """Bucketed 6:4:2 negatives from ALL 925 training queries."""
    records = []
    for case in dataset:
        query = case["entailed_fragment"]
        ground_truth = set(case["ground_truth"])
        candidates = case["candidates"]
        hybrid_top = hybrid_map.get(case["query_id"], [])

        for gt_file in ground_truth:
            if gt_file in candidates:
                records.append({"query": query, "paragraph": candidates[gt_file], "label": 1})

        negatives = [f for f in hybrid_top if f not in ground_truth and f in candidates]
        if len(negatives) > 0:
            hard = negatives[:20]
            for n in hard[:6]:
                records.append({"query": query, "paragraph": candidates[n], "label": 0})
            semi_hard = negatives[20:60]
            if semi_hard:
                sz = min(4, len(semi_hard))
                for n in random.sample(semi_hard, sz):
                    records.append({"query": query, "paragraph": candidates[n], "label": 0})
            easy = negatives[60:]
            if easy:
                sz = min(2, len(easy))
                for n in random.sample(easy, sz):
                    records.append({"query": query, "paragraph": candidates[n], "label": 0})

    return pd.DataFrame(records)

# =====================================================================
# STEP 5: TRAIN + EVALUATE ON TEST
# =====================================================================

def train_and_eval_test(config_name, model_name, use_pma, use_focal, use_bucketed, seed):
    """Train on ALL 925 training queries, evaluate on 100 test queries."""
    print(f"\n{'='*70}")
    print(f"  TRAINING: {config_name} (seed={seed})")
    print(f"{'='*70}")

    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

    tok = AutoTokenizer.from_pretrained(model_name)

    # Build training data from ALL training queries
    if use_bucketed:
        train_df = create_bucketed_train()
    else:
        train_df = create_standard_neg_train()

    hf_ds = HFDataset.from_pandas(train_df)
    def _tok_fn(ex):
        return tok(ex["query"], ex["paragraph"], padding="max_length", truncation=True, max_length=512)
    tokenized_ds = hf_ds.map(_tok_fn, batched=True).remove_columns(["query", "paragraph"])
    print(f"  Training set: {len(tokenized_ds)} pairs")

    if use_pma:
        mdl = PMACrossEncoder_T(model_name, num_labels=2, num_heads=4).to("cuda")
    else:
        mdl = CLSCrossEncoder(model_name, num_labels=2).to("cuda")

    args = TrainingArguments(
        output_dir=f"./test_abl_{config_name}_{seed}",
        num_train_epochs=4,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        learning_rate=2e-5,
        warmup_ratio=0.1,
        fp16=True,
        weight_decay=0.01,
        logging_steps=50,
        report_to="none",
        save_strategy="no",
        seed=seed,
    )

    TrainerCls = FocalTrainer_T if use_focal else CETrainer_T
    trainer = TrainerCls(model=mdl, args=args, train_dataset=tokenized_ds)
    trainer.train()

    # --- Evaluate on TEST data ---
    mdl.eval()
    preds = {}
    for case in tqdm(test_dataset, desc=f"Test Eval {config_name} s{seed}"):
        q_id = case["query_id"]
        query = case["entailed_fragment"]
        gt = set(case["ground_truth"])
        candidates = case["candidates"]
        hybrid_top = test_hybrid_map.get(q_id, [])
        valid_fnames = [f for f in hybrid_top if f in candidates]

        if not valid_fnames:
            preds[q_id] = {"scored": [], "ground_truth": gt}
            continue

        texts = [candidates[f] for f in valid_fnames]
        all_probs = []

        for i in range(0, len(texts), 8):
            batch = texts[i:i+8]
            inputs = tok([query]*len(batch), batch, padding=True, truncation=True,
                         max_length=512, return_tensors="pt").to("cuda")
            with torch.no_grad(), torch.amp.autocast('cuda'):
                logits = mdl(**inputs).logits
                probs = F.softmax(logits, dim=-1)[:, 1].cpu().tolist()
                all_probs.extend(probs)

        scored = sorted(zip(valid_fnames, all_probs), key=lambda x: x[1], reverse=True)
        preds[q_id] = {"scored": scored, "ground_truth": gt}

    del mdl, trainer, tokenized_ds, hf_ds, train_df
    gc.collect(); torch.cuda.empty_cache()

    return preds

def eval_top1(preds):
    total_tp, total_ret, total_rel = 0, 0, 0
    for q_id, data in preds.items():
        if not data["scored"]:
            total_rel += len(data["ground_truth"])
            continue
        predicted = {data["scored"][0][0]}
        total_tp += len(predicted & data["ground_truth"])
        total_ret += len(predicted)
        total_rel += len(data["ground_truth"])
    p = total_tp / total_ret if total_ret > 0 else 0
    r = total_tp / total_rel if total_rel > 0 else 0
    f1 = 2*p*r / (p+r) if (p+r) > 0 else 0
    return p, r, f1

def eval_3gate(preds):
    total_tp, total_ret, total_rel = 0, 0, 0
    for q_id, data in preds.items():
        predicted = set(safe_max_gap(data["scored"]))
        total_tp += len(predicted & data["ground_truth"])
        total_ret += len(predicted)
        total_rel += len(data["ground_truth"])
    p = total_tp / total_ret if total_ret > 0 else 0
    r = total_tp / total_rel if total_rel > 0 else 0
    f1 = 2*p*r / (p+r) if (p+r) > 0 else 0
    return p, r, f1

def ensemble_preds(pred_list):
    all_qids = list(pred_list[0].keys())
    ensembled = {}
    for q_id in all_qids:
        gt = pred_list[0][q_id]["ground_truth"]
        all_files = set()
        for preds in pred_list:
            all_files.update([f for f, p in preds[q_id]["scored"]])
        avg_scores = {}
        for fname in all_files:
            scores = []
            for preds in pred_list:
                score_dict = {f: p for f, p in preds[q_id]["scored"]}
                scores.append(score_dict.get(fname, 0.0))
            avg_scores[fname] = np.mean(scores)
        scored = sorted(avg_scores.items(), key=lambda x: x[1], reverse=True)
        ensembled[q_id] = {"scored": scored, "ground_truth": gt}
    return ensembled

# =====================================================================
# STEP 6: RUN ALL 6 CONFIGS
# =====================================================================
ablation_results = {}

# Config 1: Vanilla BERT [CLS]
preds_1 = train_and_eval_test("C1_VanillaBERT", "bert-base-uncased",
    use_pma=False, use_focal=False, use_bucketed=False, seed=42)
p, r, f1 = eval_top1(preds_1)
ablation_results["1. Vanilla BERT [CLS]"] = (p, r, f1)
print(f"  -> P={p:.4f} R={r:.4f} F1={f1:.4f}")
del preds_1; gc.collect(); torch.cuda.empty_cache()

# Config 2: LegalBERT [CLS]
preds_2 = train_and_eval_test("C2_LegalBERT", "nlpaueb/legal-bert-base-uncased",
    use_pma=False, use_focal=False, use_bucketed=False, seed=42)
p, r, f1 = eval_top1(preds_2)
ablation_results["2. LegalBERT [CLS]"] = (p, r, f1)
print(f"  -> P={p:.4f} R={r:.4f} F1={f1:.4f}")
del preds_2; gc.collect(); torch.cuda.empty_cache()

# Config 3: LegalBERT + PMA (CE loss, standard negs)
preds_3 = train_and_eval_test("C3_PMA", "nlpaueb/legal-bert-base-uncased",
    use_pma=True, use_focal=False, use_bucketed=False, seed=42)
p, r, f1 = eval_top1(preds_3)
ablation_results["3. LegalBERT + PMA"] = (p, r, f1)
print(f"  -> P={p:.4f} R={r:.4f} F1={f1:.4f}")
del preds_3; gc.collect(); torch.cuda.empty_cache()

# Config 4: LegalBERT + PMA + Focal (bucketed negs, single seed)
preds_4 = train_and_eval_test("C4_Focal", "nlpaueb/legal-bert-base-uncased",
    use_pma=True, use_focal=True, use_bucketed=True, seed=42)
p, r, f1 = eval_top1(preds_4)
ablation_results["4. LegalBERT + PMA + Focal"] = (p, r, f1)
print(f"  -> P={p:.4f} R={r:.4f} F1={f1:.4f}")

# Config 5: 3-Seed Ensemble (Top-1)
preds_43 = train_and_eval_test("C5_s43", "nlpaueb/legal-bert-base-uncased",
    use_pma=True, use_focal=True, use_bucketed=True, seed=43)
preds_44 = train_and_eval_test("C5_s44", "nlpaueb/legal-bert-base-uncased",
    use_pma=True, use_focal=True, use_bucketed=True, seed=44)

ens = ensemble_preds([preds_4, preds_43, preds_44])
p, r, f1 = eval_top1(ens)
ablation_results["5. + Ensemble (Top-1)"] = (p, r, f1)
print(f"  -> P={p:.4f} R={r:.4f} F1={f1:.4f}")

# Config 6: Full Pipeline (3-Gate on ensemble)
p, r, f1 = eval_3gate(ens)
ablation_results["6. Full Pipeline (3-Gate)"] = (p, r, f1)
print(f"  -> P={p:.4f} R={r:.4f} F1={f1:.4f}")

del preds_4, preds_43, preds_44, ens
gc.collect(); torch.cuda.empty_cache()

# =====================================================================
# FINAL TABLE
# =====================================================================
print("\n" + "=" * 78)
print("  COMPONENT-WISE ABLATION TABLE (TEST SET — 100 QUERIES)")
print("=" * 78)
print(f"{'Configuration':<40} | {'Precision':>9} {'Recall':>8} {'F1':>8}")
print("-" * 78)
for config, (p, r, f1) in ablation_results.items():
    print(f"{config:<40} | {p:>9.4f} {r:>8.4f} {f1:>8.4f}")
print("=" * 78)
print("\nPAPER: Use this table in Section 5 (Component-Wise Ablation).")
print("       All results are micro-averaged on the official COLIEE 2026 test set.")
